# 🎙️ OmniVoice Backend
**GPU сервер для Voice Clone Bot**

1. Запусти все ячейки по порядку
2. Скопируй ngrok URL в конце
3. Отправь боту: `/seturl https://xxxx.ngrok-free.app/generate`
4. Готово — бот работает пока Colab открыт

⚠️ **Runtime → Change runtime type → T4 GPU**

In [ ]:
# ============================================
# 1. Установка зависимостей (~2 мин)
# ============================================
!pip install -q omnivoice soundfile fastapi uvicorn pyngrok nest_asyncio

In [ ]:
# ============================================
# 2. Загрузка модели OmniVoice (~3 мин)
# ============================================
import torch
from omnivoice import OmniVoice

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)
print("✅ Модель загружена на GPU")

In [ ]:
# ============================================
# 3. FastAPI сервер
# ============================================
import base64
import io
import subprocess
import tempfile
import soundfile as sf
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

class TTSRequest(BaseModel):
    audio_base64: str
    text: str

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0)}

@app.post("/generate")
async def generate(req: TTSRequest):
    try:
        audio_bytes = base64.b64decode(req.audio_base64)

        with tempfile.NamedTemporaryFile(suffix=".ogg", delete=False) as f:
            f.write(audio_bytes)
            input_path = f.name

        ref_path = input_path.replace(".ogg", ".wav")
        subprocess.run(
            ["ffmpeg", "-y", "-i", input_path, "-ar", "24000", "-ac", "1", ref_path],
            capture_output=True,
        )

        audio = model.generate(
            text=req.text,
            ref_audio=ref_path,
        )

        buf = io.BytesIO()
        sf.write(buf, audio[0], 24000, format="WAV")
        buf.seek(0)
        wav_bytes = buf.read()

        return {
            "audio_base64": base64.b64encode(wav_bytes).decode(),
            "duration_sec": round(len(audio[0]) / 24000, 2),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✅ FastAPI готов")

In [ ]:
# ============================================
# 4. Запуск сервера + ngrok туннель
# ============================================
# ⚠️ Вставь свой ngrok authtoken (бесплатный)
# Получить: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # <-- вставь сюда

import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading

nest_asyncio.apply()

# Настройка ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Запуск uvicorn в фоне
thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"},
    daemon=True,
)
thread.start()

# Открываем туннель
public_url = ngrok.connect(8000)

print("\n" + "="*50)
print("🚀 СЕРВЕР ЗАПУЩЕН")
print(f"🔗 URL: {public_url}")
print(f"\n📋 Команда для бота:")
print(f"/seturl {public_url}/generate")
print("="*50)

In [ ]:
# ============================================
# 5. Быстрый тест (опционально)
# ============================================
import requests

# Проверка здоровья
r = requests.get(f"{public_url}/health")
print(r.json())

## ✅ Готово!
1. Скопируй команду `/seturl ...` выше
2. Отправь её боту в Telegram
3. Бот работает пока этот Colab открыт

**Если Colab отключился** — запусти заново все ячейки, отправь новый `/seturl`